In [216]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from scipy.signal import welch
from scipy.stats import entropy

### Load subject metadata

In [217]:
data_dir = Path('D:/DREAMT')
subject_metadata = pd.read_csv(Path(data_dir, 'participant_info.csv'))

In [218]:
subject_metadata

,SID,AGE,GENDER,BMI,OAHI,AHI,Mean_SaO2,Arousal Index,MEDICAL_HISTORY,Sleep_Disorders
0,S002,65.90,M,27.0,19,19,91%,98,"Asthma, Body Pain, GERD, Hypertension, Sleep A...",OSA
1,S003,29.38,F,51.0,34,37,95%,28,NaN,"snoring, sleep apnea, difficulty breathing, sn..."
2,S004,55.66,F,41.0,63,99,89%,109,"Arrhythmia, Body Pain, Depression, Dyspnea, GERD",difficulty breathing
3,S005,49.12,F,43.0,19,20,95%,28,"Asthma, Body Pain, Depression, Diabetes, Dyspn...",OSA
4,S006,36.91,F,22.0,4,5,97%,34,"Depression, Sleep Apnea",OSA
...,...,...,...,...,...,...,...,...,...,...
95,S099,59.92,M,26.0,31,31,94%,68,"Body Pain, CAD, GERD, Sleep Apnea","OSA, difficulty breathing, morning headaches"
96,S100,59.89,F,28.0,25,25,95%,20,"Body Pain, GERD","snoring, RLS, sleep apnea, EDS"
97,S101,38.02,F,39.0,1,2,99%,20,"Diabetes, Hypertension",H/O OSA
98,S102,57.44,M,26.0,18,24,95%,43,"Asthma, Depression, GERD, Migraine, Sleep Apnea","RLS, OSA"


### Clean subject metadata 
check missing, Mean_SaO2 in numbers

In [219]:
# all column names are lower and snake style
subject_metadata.columns = subject_metadata.columns.str.lower().str.replace(' ', '_')

In [220]:
# print column with null
for column, count in subject_metadata.isnull().sum().items():
    if count > 0:
        print(f"Column: {column}, Null Count: {count}")

Column: medical_history, Null Count: 13
Column: sleep_disorders, Null Count: 6


In [221]:
# remove % and convert to integers for mean_saO2
subject_metadata['mean_sao2'] = subject_metadata['mean_sao2'].apply(lambda row: int(row[:-1]))

# convert nan to empty list, str of conditions to list of str
subject_metadata['sleep_disorders'] = (
    subject_metadata['sleep_disorders']
    .fillna('')                              # replace NaN with empty string
    .apply(lambda x: [s.strip() for s in x.split(',')] if x else [])
)

subject_metadata['medical_history'] = (
    subject_metadata['medical_history']
    .fillna('')                              # replace NaN with empty string
    .apply(lambda x: [s.strip() for s in x.split(',')] if x else [])
)

One hot encode Gender, medical history, sleep disorders

In [222]:
# Keep keys normalized (lowercase).
SYN2CAN = {
# sleep apnea family
"sleep apnea": "osa",
"sleep apena": "osa", # just in case
"osa": "osa",
"h/o osa": "osa",
# snoring family
"snore": "snoring",
"snoring": "snoring",
"snorts": "snoring",
"snort": "snoring",
# breathing issues
"difficulty breathing": "dyspnea",
"diffifulty breathing" : "dyspnea",
"difficulty staying asleep": "insomnia",
# teeth grinding
"bruxism": "bruxism",
"grinds teeth": "bruxism",
"grind teeth": "bruxism",
# fatigue / sleepiness
"fatigue": "fatigue",
"chronic fatigue": "fatigue",
"hypersomnia": "hypersomnia",
"eds": "eds", # excessive daytime sleepiness
"rbd": "rbd",
"rls": "rls",
# headaches
"headache": "migraine",
"morning headaches": "migraine",
"migraine": "migraine",
# none
"none": None,
# medical history (normalize to snake-like)
"asthma": "asthma",
"body pain": "body_pain",
"gerd": "gerd",
"anxiety": "anxiety",
"depression": "depression",
"dyspnea": "dyspnea",
"diabetes": "diabetes",
"arrhythmia": "arrhythmia",
"cad": "cad",
"hypertension": "hypertension",
"mci": "mci"
}

def normalize_med_history(history_list, synmap):
    """
    Takes a list of strings in medical_history,
    normalizes via SYN2CAN, removes duplicates and None.
    """
    if not isinstance(history_list, list):
        return []

    normalized = []
    for h in history_list:
        key = h.lower().strip()
        mapped = synmap.get(key, key)   # default: keep same key
        if mapped is not None:
            normalized.append(mapped)

    # dedupe while preserving order
    return list(dict.fromkeys(normalized))

subject_metadata["medical_history_norm"] = subject_metadata["medical_history"].apply(
    lambda x: normalize_med_history(x, SYN2CAN)
)
subject_metadata["sleep_disorders_norm"] = subject_metadata["sleep_disorders"].apply(
    lambda x: normalize_med_history(x, SYN2CAN)
)

In [223]:
gender_encoder = OneHotEncoder(drop='first', sparse_output=False, dtype=int)
gender_ohe = gender_encoder.fit_transform(subject_metadata[['gender']])
df_gender = pd.DataFrame(gender_ohe, columns=gender_encoder.get_feature_names_out(['gender']))
subject_metadata = subject_metadata.join(df_gender)
subject_metadata = subject_metadata.drop('gender', axis=1)

mlb_sleep = MultiLabelBinarizer()
sleep_ohe = mlb_sleep.fit_transform(subject_metadata['sleep_disorders_norm'])
df_sleep_ohe = pd.DataFrame(sleep_ohe, columns=[f"sd_{d.replace(' ', '_')}" for d in mlb_sleep.classes_])
subject_metadata = subject_metadata.join(df_sleep_ohe)
subject_metadata = subject_metadata.drop('sleep_disorders_norm', axis=1)
subject_metadata = subject_metadata.drop('sleep_disorders', axis=1)

mlb_med = MultiLabelBinarizer()
med_ohe = mlb_med.fit_transform(subject_metadata['medical_history_norm'])
df_med_ohe = pd.DataFrame(med_ohe, columns=[f"med_{d.replace(' ', '_')}" for d in mlb_med.classes_])
subject_metadata = subject_metadata.join(df_med_ohe)
subject_metadata = subject_metadata.drop('medical_history_norm', axis=1)
subject_metadata = subject_metadata.drop('medical_history', axis=1)

In [224]:
# all column names are lower and snake style
subject_metadata.columns = subject_metadata.columns.str.lower().str.replace(' ', '_')
print(subject_metadata.shape)
subject_metadata.head()

(100, 33)


,sid,age,bmi,oahi,ahi,mean_sao2,arousal_index,gender_m,sd_bruxism,sd_dyspnea,...,med_asthma,med_body_pain,med_cad,med_depression,med_diabetes,med_dyspnea,med_gerd,med_hypertension,med_migraine,med_osa
0,S002,65.90,27.0,19,19,91,98,1,0,0,...,1,1,0,0,0,0,1,1,0,1
1,S003,29.38,51.0,34,37,95,28,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,S004,55.66,41.0,63,99,89,109,0,0,1,...,0,1,0,1,0,1,1,0,0,0
3,S005,49.12,43.0,19,20,95,28,0,0,0,...,1,1,0,1,1,1,1,0,0,1
4,S006,36.91,22.0,4,5,97,34,0,0,0,...,0,0,0,1,0,0,0,0,0,1


In [225]:
# save preprocessed metadata
# make proceesed folder if not exists
processed_dir = Path(data_dir, 'processed')
processed_dir.mkdir(exist_ok=True)

subject_metadata.to_csv(Path(processed_dir, 'participant_info_preprocessed.csv'), index=False)

### Load Data 100Hz

In [199]:
sid = 'S005' # 'S002'
data_100hz = pd.read_csv(Path(data_dir, 'data_100Hz', f'{sid}_PSG_df_updated.csv'))

In [200]:
data_100hz.head()

,TIMESTAMP,C4-M1,F4-M1,O2-M1,Fp1-O2,T3 - CZ,CZ - T4,CHIN,E1,E2,...,ACC_Z,TEMP,EDA,HR,IBI,Sleep_Stage,Obstructive_Apnea,Central_Apnea,Hypopnea,Multiple_Events
0,0.00,0.000003,1.513695e-06,-3.418021e-07,9.277485e-07,9.277485e-07,3.418021e-07,1.904326e-06,0.000002,6.347753e-07,...,31.0,33.84,0.260079,87.38,0.75,P,NaN,NaN,NaN,NaN
1,0.01,0.000002,7.324331e-07,-8.300908e-07,1.806668e-06,4.394598e-07,8.300908e-07,1.123064e-06,0.000002,-7.324331e-07,...,31.0,33.84,0.260079,87.38,0.75,P,NaN,NaN,NaN,NaN
2,0.02,0.000002,4.394598e-07,-1.123064e-06,2.587930e-06,-1.464866e-07,1.416037e-06,2.001984e-06,0.000002,-2.441444e-07,...,31.0,33.84,0.260079,87.38,0.75,P,NaN,NaN,NaN,NaN
3,0.03,0.000002,1.123064e-06,-2.441444e-07,2.001984e-06,-2.441444e-07,2.099641e-06,-2.441444e-07,0.000002,1.025406e-06,...,31.0,33.84,0.260079,87.38,0.75,P,NaN,NaN,NaN,NaN
4,0.04,0.000003,1.318379e-06,-4.882887e-08,9.277485e-07,2.441444e-07,1.904326e-06,-1.709010e-06,0.000001,-5.371176e-07,...,31.0,33.84,0.260079,87.38,0.75,P,NaN,NaN,NaN,NaN


### Clean data_100Hz

remove missing values

In [201]:
# all column names are lowercase
data_100hz.columns = data_100hz.columns.str.lower().str.replace(' ','')
data_100hz.columns

Index(['timestamp', 'c4-m1', 'f4-m1', 'o2-m1', 'fp1-o2', 't3-cz', 'cz-t4',
       'chin', 'e1', 'e2', 'ecg', 'lat', 'rat', 'snore', 'ptaf', 'flow',
       'thorax', 'abdomen', 'sao2', 'bvp', 'acc_x', 'acc_y', 'acc_z', 'temp',
       'eda', 'hr', 'ibi', 'sleep_stage', 'obstructive_apnea', 'central_apnea',
       'hypopnea', 'multiple_events'],
      dtype='str')

In [202]:
# print column with null
null_columns = []
for column, count in data_100hz.isnull().sum().items():
    if count > 0:
        print(f"Column: {column}, Null Count: {count}")
        null_columns.append(column)

Column: obstructive_apnea, Null Count: 3112300
Column: central_apnea, Null Count: 3108998
Column: hypopnea, Null Count: 2935250
Column: multiple_events, Null Count: 3107496


In [203]:
# NA in obstructive_apnea, central_apnea, hypopnea, multiple_events 
# change NA to 0
for col in null_columns:
    data_100hz[col] = data_100hz[col].apply(lambda x: 0 if pd.isna(x) else 1)

remove stage P(preparation) for now

In [204]:
# length(min) of each stage
data_100hz['sleep_stage'].value_counts() / 6000

sleep_stage
N2    240.000000
P     152.217167
R      58.000000
W      46.499500
N1     19.500000
N3      2.500000
Name: count, dtype: float64

In [205]:
data_100hz = data_100hz[data_100hz.sleep_stage != 'P'].reset_index(drop=True)
# length(min) of each stage
data_100hz['sleep_stage'].value_counts() / 6000

sleep_stage
N2    240.0000
R      58.0000
W      46.4995
N1     19.5000
N3      2.5000
Name: count, dtype: float64

In [206]:
data_100hz

,timestamp,c4-m1,f4-m1,o2-m1,fp1-o2,t3-cz,cz-t4,chin,e1,e2,...,acc_z,temp,eda,hr,ibi,sleep_stage,obstructive_apnea,central_apnea,hypopnea,multiple_events
0,9133.03,-1.513695e-06,-7.324331e-07,-3.418021e-07,4.394598e-07,0.000006,-5.224689e-06,1.025406e-06,-5.224689e-06,-1.806668e-06,...,31.0,33.84,0.260079,87.38,0.7500,W,0,0,0,0
1,9133.04,-3.369192e-06,-3.955138e-06,-4.541085e-06,6.347753e-07,0.000006,-4.052796e-06,-2.880903e-06,-7.568475e-06,-2.880903e-06,...,31.0,33.84,0.260079,87.38,0.7500,W,0,0,0,0
2,9133.05,-1.157244e-05,-7.959106e-06,-6.396582e-06,-9.277485e-07,0.000010,-1.020523e-05,-2.880903e-06,-3.564508e-06,1.806668e-06,...,31.0,33.84,0.260079,87.38,0.7500,W,0,0,0,0
3,9133.06,-1.127947e-05,-6.884871e-06,-3.466850e-06,-5.127031e-06,0.000009,-1.499046e-05,-1.123064e-06,-4.052796e-06,4.394598e-07,...,31.0,33.84,0.260079,87.38,0.7500,W,0,0,0,0
4,9133.07,1.611353e-06,-2.978561e-06,3.369192e-06,-1.030289e-05,0.000006,-7.177844e-06,8.300908e-07,-5.371176e-07,2.001984e-06,...,31.0,33.84,0.260079,87.38,0.7500,W,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2198992,31122.95,-1.806668e-06,-4.882887e-08,1.709010e-06,-1.220722e-06,0.000003,-2.099641e-06,4.394598e-07,4.052796e-06,-2.001984e-06,...,-20.0,35.97,0.531688,65.42,0.8125,N2,0,0,1,0
2198993,31122.96,-1.123064e-06,2.441444e-07,1.904326e-06,-1.416037e-06,0.000005,-2.294957e-06,-1.464866e-07,3.271534e-06,-2.587930e-06,...,-20.0,35.97,0.531688,65.42,0.8125,N2,0,0,1,0
2198994,31122.97,-4.882887e-08,1.464866e-07,2.587930e-06,-2.392615e-06,0.000004,-6.347753e-07,4.882887e-08,1.318379e-06,-4.736400e-06,...,-20.0,35.97,0.531688,65.42,0.8125,N2,0,0,1,0
2198995,31122.98,1.464866e-07,-1.318379e-06,3.369192e-06,-5.420005e-06,0.000001,2.587930e-06,-2.587930e-06,-4.834058e-06,-9.228656e-06,...,-20.0,35.97,0.531688,65.42,0.8125,N2,0,0,1,0


# Feature Engineering

In [207]:
data_100hz.columns

Index(['timestamp', 'c4-m1', 'f4-m1', 'o2-m1', 'fp1-o2', 't3-cz', 'cz-t4',
       'chin', 'e1', 'e2', 'ecg', 'lat', 'rat', 'snore', 'ptaf', 'flow',
       'thorax', 'abdomen', 'sao2', 'bvp', 'acc_x', 'acc_y', 'acc_z', 'temp',
       'eda', 'hr', 'ibi', 'sleep_stage', 'obstructive_apnea', 'central_apnea',
       'hypopnea', 'multiple_events'],
      dtype='str')

In [208]:
vars_to_consider.keys()

dict_keys(['timestamp', 'c4-m1', 'f4-m1', 'o2-m1', 'fp1-o2', 't3-cz', 'cz-t4', 'e1', 'e2', 'chin', 'lat', 'rat', 'ptaf', 'flow', 'thorax', 'abdomen', 'obstructive_apnea', 'central_apnea', 'hypopnea', 'multiple_events', 'bvp', 'ibi', 'eda', 'temp', 'hr', 'snore', 'sao2', 'acc_x', 'acc_y', 'acc_z'])

In [209]:
processed_data = pd.DataFrame()
# change column names with following dict
vars_to_consider = {
    'timestamp': 'timestamp',
    'c4-m1': 'eeg_c4',
    'f4-m1': 'eeg_f4',
    'o2-m1': 'eeg_o2',
    'fp1-o2': 'eeg_fp1',
    't3-cz': 'eeg_t3',
    'cz-t4': 'eeg_cz',  
    'e1': 'eog_e1',
    'e2': 'eog_e2',
    'chin': 'emg_chin',
    'lat': 'emg_lat',
    'rat': 'emg_rat',
    'ptaf': 'resp_ptaf',
    'flow': 'resp_flow',
    'thorax': 'resp_thorax',
    'abdomen': 'resp_abdomen',
    'obstructive_apnea': 'apnea_obstructive',
    'central_apnea': 'apnea_central',
    'hypopnea': 'apnea_hypopnea',
    'multiple_events': 'apnea_mixed',
    'bvp': 'bvp',
    'ibi': 'ibi',
    'eda': 'eda',
    'temp': 'temp',
    'hr': 'hr',
    'snore': 'snore',
    'sao2': 'sao2',
    'acc_x': 'acc_x',
    'acc_y': 'acc_y',
    'acc_z': 'acc_z'
    }


for old_var, new_var in vars_to_consider.items():
    if old_var in data_100hz.columns:
        processed_data[new_var] = data_100hz[old_var]


In [ ]:

# EEG / EOG / EMG bands
EEG_BANDS = {"delta": (0.5,4), "theta":(4,8), "alpha":(8,12), "sigma":(12,15),
             "beta":(15,30),"gamma":(30,45),"high_gamma":(45,80)}
EOG_BANDS = {"slow":(0.1,1),"delta":(1,4),"theta":(4,8),"high":(8,15)}
EMG_BANDS = {"low":(0.5,10),"medium":(10,30),"high":(30,100)}

# -----------------------------
# Helper functions
# -----------------------------
def bandpower(data, fs, band):
    low, high = band
    freqs, psd = welch(data, fs=fs, nperseg=min(4*fs,len(data)))
    idx = np.logical_and(freqs>=low, freqs<=high)
    return np.trapezoid(psd[idx], freqs[idx])

def extract_basic_stats(signal):
    return {
        'mean': np.mean(signal),
        'std': np.std(signal),
        'min': np.min(signal),
        'max': np.max(signal),
        'range': np.max(signal)-np.min(signal),
        'slope': 0 if np.std(signal) == 0 else np.polyfit(np.arange(len(signal)), signal,1)[0]
    }

def compute_acc_magnitude(acc_x,acc_y,acc_z):
    return np.sqrt(acc_x**2 + acc_y**2 + acc_z**2)

def compute_hrv_metrics(ibi_ms):
    """
    ibi_ms: IBI in milliseconds
    """
    if len(ibi_ms)<2:
        return {'sdnn': np.nan, 'rmssd': np.nan, 'pnn50': np.nan}
    diff = np.diff(ibi_ms)
    sdnn = np.std(ibi_ms)
    rmssd = np.sqrt(np.mean(diff**2))
    pnn50 = np.sum(np.abs(diff)>50)/len(diff)  # fraction >50ms
    return {'sdnn': sdnn, 'rmssd': rmssd, 'pnn50': pnn50}

def extract_eda_phasic(eda_signal, threshold=0.01):
    """Count phasic peaks by thresholding derivative"""
    if len(eda_signal)<2:
        return {'eda_scr_count':0,'eda_scr_mean_amp':0}
    diff = np.diff(eda_signal)
    peaks = diff > threshold
    count = np.sum(peaks)
    mean_amp = np.mean(diff[peaks]) if count>0 else 0
    return {'eda_scr_count': int(count), 'eda_scr_mean_amp': mean_amp}

# -----------------------------
# Main extraction function
# -----------------------------
def extract_full_multimodal(df, signals, epoch_sec=5, fs=100):
    start_time = df['timestamp'].min()
    end_time = df['timestamp'].max()
    n_epochs = int((end_time-start_time)//epoch_sec)
    extended_time = 25
    
    ## Calculate acceleration magnitude
    if all(x in df.columns for x in ['acc_x','acc_y','acc_z']) and len(df)>0:
        df['acc_magnitude'] = compute_acc_magnitude(df['acc_x'].values,
                                                    df['acc_y'].values,
                                                    df['acc_z'].values)
    all_features = []        

    for i in range(n_epochs):
        epoch_start = start_time + i*epoch_sec
        epoch_end = epoch_start + epoch_sec
        epoch_df = df[(df['timestamp']>=epoch_start)&(df['timestamp']<epoch_end)]
        extended_epoch_df = df[(df['timestamp']>=epoch_start-extended_time)&(df['timestamp']<epoch_end)]
        epoch_feats = {'epoch_id':i, 'epoch_start':epoch_start, 'epoch_end':epoch_end}
        
        # -----------------------------
        # Time-domain features
        # -----------------------------
        for sig in signals:
            if sig in ['acc_x','acc_y','acc_z']:
                continue
            if sig in epoch_df.columns and len(epoch_df[sig])>0:
                epoch_feats.update({f'{sig}_{k}':v for k,v in extract_basic_stats(epoch_df[sig].values).items()})
        
        
        # -----------------------------
        # EEG bandpowers
        # -----------------------------
        for sig in signals:
            if sig.lower().startswith('eeg') and sig in epoch_df.columns:
                eeg_signal = extended_epoch_df[sig].values
                for band_name, band_range in EEG_BANDS.items():
                    epoch_feats[f'{sig}_bp_{band_name}'] = bandpower(eeg_signal, fs, band_range)
        
        # -----------------------------
        # EOG bandpowers
        # -----------------------------
        for sig in signals:
            if sig.lower().startswith('eog') and sig in epoch_df.columns:
                eog_signal = extended_epoch_df[sig].values
                for band_name, band_range in EOG_BANDS.items():
                    epoch_feats[f'{sig}_bp_{band_name}'] = bandpower(eog_signal, fs, band_range)
        
        # -----------------------------
        # EMG bandpowers
        # -----------------------------
        for sig in signals:
            if sig.lower().startswith('emg') and sig in epoch_df.columns:
                emg_signal = extended_epoch_df[sig].values
                for band_name, band_range in EMG_BANDS.items():
                    epoch_feats[f'{sig}_bp_{band_name}'] = bandpower(emg_signal, fs, band_range)
        
        # -----------------------------
        # IBI / HRV metrics
        # -----------------------------
        if 'ibi' in epoch_df.columns and len(epoch_df['ibi'])>1:
            ibi_ms = epoch_df['ibi'].values
            hrv = compute_hrv_metrics(ibi_ms)
            epoch_feats.update({f'ibi_{k}':v for k,v in hrv.items()})
        
        # -----------------------------
        # EDA phasic peaks
        # -----------------------------
        if 'eda' in epoch_df.columns and len(epoch_df['eda'])>1:
            eda_feats = extract_eda_phasic(epoch_df['eda'].values)
            epoch_feats.update(eda_feats)
        
        # -----------------------------
        # Cross-signal features
        # -----------------------------
        # HR/BVP correlation
        if 'hr' in epoch_df.columns and 'bvp' in epoch_df.columns and len(epoch_df)>1:
            hr = epoch_df['hr'].values
            bvp = epoch_df['bvp'].values
            if np.std(hr) > 0 and np.std(bvp) > 0:
                epoch_feats['hr_bvp_corr'] = np.corrcoef(hr,bvp)[0,1]
            else:
                epoch_feats['hr_bvp_corr'] = np.nan
        
        # ACC + EDA arousal proxy
        if 'eda' in epoch_df.columns and 'acc_magnitude' in epoch_df.columns and len(epoch_df)>1:
            acc_mag = epoch_df['acc_magnitude'].values
            # correlation as simple arousal proxy
            if np.std(acc_mag)>0 and np.std(epoch_df['eda'].values)>0:
                epoch_feats['acc_eda_corr'] = np.corrcoef(acc_mag, epoch_df['eda'].values)[0,1]
            else:
                epoch_feats['acc_eda_corr'] = np.nan 
        
        all_features.append(epoch_feats)
    
    return pd.DataFrame(all_features)

In [215]:
signals = list(vars_to_consider.values())
signals.remove('timestamp')  # we don't want to extract features from timestamp


features_df = extract_full_multimodal(
    df=processed_data,
    signals=signals,
    epoch_sec=5,
)

print(features_df.shape)
features_df.head()

(4397, 225)


,epoch_id,epoch_start,epoch_end,eeg_c4_mean,eeg_c4_std,eeg_c4_min,eeg_c4_max,eeg_c4_range,eeg_c4_slope,eeg_f4_mean,...,emg_rat_bp_low,emg_rat_bp_medium,emg_rat_bp_high,ibi_sdnn,ibi_rmssd,ibi_pnn50,eda_scr_count,eda_scr_mean_amp,hr_bvp_corr,acc_eda_corr
0,0,9133.03,9138.03,-5.455161e-07,0.000007,-0.000032,0.000022,0.000053,6.028274e-09,-0.000003,...,1.470885e-13,1.213987e-13,7.645721e-14,0.010097,0.001851,0.0,0,0.0,-0.047456,0.070481
1,1,9138.03,9143.03,4.332097e-07,0.000009,-0.000024,0.000033,0.000057,1.359185e-08,0.000001,...,1.840027e-13,2.059829e-13,7.148550e-14,0.007119,0.001399,0.0,0,0.0,-0.024225,0.106972
2,2,9143.03,9148.03,1.361349e-07,0.000008,-0.000026,0.000024,0.000050,-2.398504e-09,0.000003,...,1.309245e-12,1.714546e-12,1.871017e-13,0.014027,0.002617,0.0,0,0.0,0.057626,0.159981
3,3,9148.03,9153.03,2.871138e-07,0.000010,-0.000028,0.000028,0.000056,-1.269840e-09,-0.000001,...,1.048375e-12,1.255038e-12,1.697074e-13,0.015666,0.003049,0.0,0,0.0,0.006178,0.019217
4,4,9153.03,9158.03,-1.039078e-06,0.000014,-0.000050,0.000038,0.000089,5.062973e-10,-0.000002,...,8.760060e-13,1.054504e-12,1.506875e-13,0.019325,0.003567,0.0,0,0.0,-0.110078,0.074976
